# 📊 Buổi 3 — Exploratory Data Analysis (EDA)

---

> **📚 Khóa học:** Machine Learning thực chiến — Từ Zero đến Kaggle  
> **📅 Buổi:** 3 / 9  
> **⏱️ Thời lượng:** ~75 phút  
> **🎯 Mục tiêu:** Khám phá 80+ features, hiểu mối quan hệ với `SalePrice`, phát hiện missing values và outliers

---

## 🗺️ Nội dung buổi học

| # | Chủ đề | Loại | Thời gian |
|---|--------|------|-----------|
| 1️⃣ | Phân tích dữ liệu thiếu (Missing Values) | 📖 + 💻 | ~20 phút |
| 2️⃣ | Phân tích biến số (Numerical Features) | 📖 + 💻 | ~20 phút |
| 3️⃣ | Phân tích biến phân loại (Categorical Features) | 📖 + 💻 | ~20 phút |
| 4️⃣ | Ma trận tương quan & Multicollinearity | 📖 + 💻 | ~15 phút |

---

## 🔗 Kết Nối Với Buổi 2

Ở **Buổi 2**, bạn đã:
- ✅ Load dataset thành công: Train (1,460 × 81), Test (1,459 × 80)
- ✅ Khám phá cấu trúc cơ bản: 36 numerical + 43 categorical features
- ✅ Phát hiện `SalePrice` bị skewed → áp dụng `log1p` transform
- ✅ Tạo biến `y` (log-transformed target) sẵn sàng cho modeling

**Buổi 3 này**, chúng ta sẽ đi sâu hơn vào **từng feature** để trả lời câu hỏi:

```
❓ Feature nào ảnh hưởng nhiều nhất đến giá nhà?
❓ Cột nào bị thiếu dữ liệu và thiếu bao nhiêu %?
❓ Có outliers (giá trị bất thường) nào cần xử lý không?
❓ Các features có "trùng thông tin" với nhau không?
```

---

## 🤔 EDA Là Gì? Tại Sao Quan Trọng?

**EDA (Exploratory Data Analysis)** — Phân tích dữ liệu khám phá — là bước **bắt buộc** trong mọi dự án ML.

```
Không có EDA → Train mô hình → Kết quả tệ → Không biết tại sao ❌

Có EDA → Hiểu dữ liệu → Xử lý đúng → Train mô hình → Kết quả tốt ✅
```

**Quy tắc trong ngành:** Các Data Scientist kinh nghiệm thường dành **60-70% thời gian** cho EDA và data preprocessing — chỉ 30-40% còn lại cho việc train model.

> 💡 **Tại sao?** Vì "garbage in, garbage out" — nếu dữ liệu đầu vào tệ, mô hình dù mạnh đến đâu cũng cho kết quả tệ!

---

> ⚠️ **Trước khi bắt đầu:** Đảm bảo bạn đã chạy toàn bộ **Buổi 2** (`buoi2_Setup.ipynb`) để các biến `train_df`, `test_df`, `y` đã được tạo ra. Nếu chưa, chạy cell đầu tiên bên dưới để setup lại.

In [ ]:
# ============================================================
# ♻️ SETUP — Chạy lại nếu kernel bị restart (kết nối Buổi 2)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Load dataset
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

# Log transform target (kết quả từ Buổi 2)
y = np.log1p(train_df['SalePrice'])

# Phân loại features
features      = train_df.drop(columns=['Id', 'SalePrice'])
numerical_cols   = features.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = features.select_dtypes(include=['object']).columns.tolist()

print("✅ Setup hoàn tất — sẵn sàng cho Buổi 3!")
print(f"   Train shape      : {train_df.shape}")
print(f"   Numerical cols   : {len(numerical_cols)}")
print(f"   Categorical cols : {len(categorical_cols)}")
print(f"   Target y range   : {y.min():.3f} → {y.max():.3f}  (log scale)")

> 💡 **Expected Output:**

![img3-1](images/img_buoi3/img3-1.png)

---

## 🕳️ Task 1 — Phân Tích Dữ Liệu Thiếu (Missing Values)

---

### 🤔 Missing Values Là Gì? Tại Sao Nguy Hiểm?

**Missing values** (giá trị thiếu/null/NaN) xảy ra khi một ô dữ liệu **không có thông tin**. Trong Pandas, chúng thường hiển thị là `NaN` (Not a Number).

```
Ví dụ thực tế trong dataset nhà:
  PoolQC = NaN  →  Ngôi nhà này KHÔNG CÓ bể bơi (không phải dữ liệu sai!)
  Alley  = NaN  →  Ngôi nhà này KHÔNG CÓ con hẻm tiếp cận
  LotFrontage = NaN → Dữ liệu thực sự BỊ THIẾU, cần điền vào
```

**Nếu không xử lý missing values:**
- Hầu hết thuật toán ML sẽ **báo lỗi** hoặc bỏ qua hàng đó
- Kết quả dự đoán bị **sai lệch** vì mô hình thiếu thông tin

---

### 📐 Hai Loại Missing Values Trong BDS

```
Loại 1: "Có ý nghĩa" (Missing = None/Không có)
   PoolQC = NaN → Nhà không có bể bơi → Điền "None" hoặc 0
   GarageType = NaN → Nhà không có garage → Điền "None"
   FireplaceQu = NaN → Nhà không có lò sưởi → Điền "None"

Loại 2: "Thực sự thiếu" (Random Missing)
   LotFrontage = NaN → Không đo được chiều dài mặt tiền
   → Điền bằng median theo Neighborhood (khu vực gần nhau thường tương đồng)
```

> 💡 **Phân biệt hai loại này là kỹ năng quan trọng** — điền sai có thể làm giảm chất lượng mô hình. Đây là lý do chúng ta cần EDA TRƯỚC khi xử lý!

---

### 🎯 Ngưỡng Xử Lý Missing Values

| 📊 % Missing | 🏷️ Mức độ | 🔧 Chiến lược phổ biến |
|---|---|---|
| < 5% | Ít | Điền bằng mean/median/mode |
| 5% – 30% | Trung bình | Điền theo nhóm (group median) hoặc model-based imputation |
| 30% – 80% | Nhiều | Tạo binary flag "có/không" + điền placeholder |
| > 80% | Rất nhiều | Cân nhắc **xóa cột** hoặc điền "None" nếu có ý nghĩa |

In [ ]:
# ============================================================
# 🕳️ TASK 1.1 — ĐẾM & PHÂN LOẠI MISSING VALUES
# ============================================================

# --- Bước 1: Tính số lượng và % missing ---
missing_count = train_df.isnull().sum()                        # Đếm NaN từng cột
missing_pct   = (missing_count / len(train_df)) * 100          # Tính phần trăm

# --- Bước 2: Tạo DataFrame tổng hợp ---
missing_df = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct'  : missing_pct
})
missing_df = missing_df[missing_df['missing_count'] > 0]       # Chỉ giữ cột có missing
missing_df = missing_df.sort_values('missing_pct', ascending=False)
missing_df['missing_pct'] = missing_df['missing_pct'].round(2)

print(f"📊 Tổng số cột trong train_df     : {train_df.shape[1]}")
print(f"📊 Số cột có missing values        : {len(missing_df)}")
print(f"📊 Số cột KHÔNG có missing values  : {train_df.shape[1] - len(missing_df)}")
print()

# --- Bước 3: Hiển thị với phân loại mức độ ---
def classify_missing(pct):
    if pct > 80:  return "🔴 Rất cao (>80%)"
    if pct > 30:  return "🟠 Cao (30-80%)"
    if pct > 5:   return "🟡 Trung bình (5-30%)"
    return                "🟢 Thấp (<5%)"

missing_df['mức_độ'] = missing_df['missing_pct'].apply(classify_missing)

print("=" * 65)
print(f"{'Cột':<20} {'Số thiếu':>10} {'% thiếu':>10}  Mức độ")
print("=" * 65)
for col, row in missing_df.iterrows():
    print(f"  {col:<18} {int(row['missing_count']):>10,} {row['missing_pct']:>9.1f}%  {row['mức_độ']}")
print("=" * 65)

**Expected Output:**

![img3-2](images/img_buoi3/img3-2.png)

---

### 🔬 Giải Thích Chi Tiết: Code Đếm Missing Values

---

#### 📌 Từng Dòng Lệnh

```python
missing_count = train_df.isnull().sum()
```

| Bước | Lệnh | Kết quả trung gian |
|---|---|---|
| 1 | `train_df.isnull()` | DataFrame boolean: `True` chỗ nào là NaN, `False` chỗ còn lại |
| 2 | `.sum()` | Cộng theo cột: `True=1`, `False=0` → đếm số NaN từng cột |

**Minh hoạ:**
```
train_df['PoolQC']:  [NaN, NaN, NaN, 'Gd', NaN, ...]
isnull():            [True, True, True, False, True, ...]
sum():               1453  ← Có 1453 giá trị NaN trong cột PoolQC
```

---

```python
missing_pct = (missing_count / len(train_df)) * 100
```

| Thành phần | Ý nghĩa |
|---|---|
| `len(train_df)` | Tổng số hàng = 1460 |
| `missing_count / 1460` | Tỷ lệ missing (0 → 1) |
| `* 100` | Chuyển sang phần trăm (0% → 100%) |

**Ví dụ:** `PoolQC`: 1453 / 1460 × 100 = **99.52%** missing

---

```python
missing_df = missing_df[missing_df['missing_count'] > 0]
```

**Boolean indexing** — lọc chỉ giữ những hàng có ít nhất 1 giá trị missing:
```
missing_df['missing_count'] > 0  →  Series boolean [True, False, True, ...]
missing_df[...]                  →  Chỉ giữ hàng True
```

---

```python
def classify_missing(pct):
    ...
missing_df['mức_độ'] = missing_df['missing_pct'].apply(classify_missing)
```

| Lệnh | Ý nghĩa |
|---|---|
| `def classify_missing(pct)` | Định nghĩa hàm phân loại mức độ theo % |
| `.apply(hàm)` | Áp dụng hàm đó cho **từng phần tử** trong cột |

**🎯 Mục đích sau cùng:** Biết được **19 cột** bị thiếu dữ liệu, mức độ nghiêm trọng của từng cột → làm cơ sở để chọn chiến lược xử lý ở Buổi 4.

In [ ]:
# ============================================================
# 🗺️ TASK 1.2 — VISUALIZE MISSING VALUES PATTERN
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🕳️ Phân Tích Missing Values', fontsize=15, fontweight='bold')

# --- Biểu đồ 1: Bar chart % missing (tất cả cột bị thiếu) ---
colors = ['#d32f2f' if p > 80 else '#f57c00' if p > 30 else '#f9a825' if p > 5 else '#388e3c'
          for p in missing_df['missing_pct']]

axes[0].barh(missing_df.index, missing_df['missing_pct'], color=colors, edgecolor='white')
axes[0].axvline(x=80, color='red',    linestyle='--', alpha=0.7, label='80% threshold')
axes[0].axvline(x=30, color='orange', linestyle='--', alpha=0.7, label='30% threshold')
axes[0].axvline(x=5,  color='green',  linestyle='--', alpha=0.7, label='5% threshold')
axes[0].set_xlabel('% Missing Values', fontsize=11)
axes[0].set_title('% Missing theo từng cột', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].invert_yaxis()

# Thêm nhãn % lên từng thanh
for i, (idx, row) in enumerate(missing_df.iterrows()):
    axes[0].text(row['missing_pct'] + 0.5, i, f"{row['missing_pct']:.1f}%",
                 va='center', fontsize=8)

# --- Biểu đồ 2: Heatmap pattern (chỉ cột >5% missing) ---
high_missing_cols = missing_df[missing_df['missing_pct'] > 5].index.tolist()
missing_matrix = train_df[high_missing_cols].isnull().astype(int)

# Chỉ lấy 200 sample để heatmap dễ nhìn
sample_matrix = missing_matrix.sample(200, random_state=42).reset_index(drop=True)

sns.heatmap(sample_matrix.T, cmap='YlOrRd', cbar=True,
            cbar_kws={'label': '1 = Missing', 'shrink': 0.8},
            ax=axes[1], linewidths=0)
axes[1].set_xlabel('Sample Index (200 ngẫu nhiên)', fontsize=11)
axes[1].set_title('Pattern Missing (>5% missing cols)', fontsize=12)
axes[1].tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

# --- Tóm tắt nhóm ---
print("\n💡 KEY INSIGHTS — Nhóm Missing Values:")
print("─" * 55)
print("🔴 >80% missing (có ý nghĩa = không có):")
for col in missing_df[missing_df['missing_pct'] > 80].index:
    print(f"   {col:<15} → Nhà KHÔNG CÓ {col.replace('QC','').replace('Qu','')}")
print()
print("🟠 30-80% missing:")
for col in missing_df[(missing_df['missing_pct'] > 30) & (missing_df['missing_pct'] <= 80)].index:
    print(f"   {col:<15} → Cân nhắc điền hoặc tạo flag")
print()
print("🟡 5-30% missing:")
for col in missing_df[(missing_df['missing_pct'] > 5) & (missing_df['missing_pct'] <= 30)].index:
    print(f"   {col:<15} → Điền theo nhóm (group median/mode)")
print()
print("🟢 <5% missing:")
for col in missing_df[missing_df['missing_pct'] <= 5].index:
    print(f"   {col:<15} → Điền median/mode đơn giản")

**Expected Output:**

![img3-3](images/img_buoi3/img3-3.png)
![img3-4](images/img_buoi3/img3-4.png)

---

### 🔬 Giải Thích Code Visualize + Chiến Lược Xử Lý

---

#### 📌 Biểu Đồ 1 — Bar Chart Ngang

```python
axes[0].barh(missing_df.index, missing_df['missing_pct'], color=colors, ...)
```

| Tham số | Ý nghĩa |
|---|---|
| `barh` | **Horizontal bar** — thanh nằm ngang (dễ đọc tên cột dài) |
| `missing_df.index` | Nhãn Y-axis = tên cột |
| `missing_df['missing_pct']` | Độ dài thanh = % missing |
| `colors` | List màu: đỏ >80%, cam >30%, vàng >5%, xanh <5% |

```python
axes[0].axvline(x=80, color='red', linestyle='--')
```

`axvline` = **Axis Vertical Line** — vẽ đường thẳng đứng tại x=80 để đánh dấu ngưỡng

```python
axes[0].invert_yaxis()
```

Lật trục Y để cột có **% missing cao nhất** hiện ra ở **trên cùng** (dễ đọc hơn)

---

#### 📌 Biểu Đồ 2 — Heatmap Pattern

```python
missing_matrix = train_df[high_missing_cols].isnull().astype(int)
```

| Bước | Kết quả |
|---|---|
| `train_df[high_missing_cols]` | Lọc chỉ những cột missing >5% |
| `.isnull()` | True/False tại mỗi ô |
| `.astype(int)` | True→1 (missing), False→0 (có data) |

```python
sample_matrix = missing_matrix.sample(200, random_state=42)
```

`sample(200)` = lấy **200 hàng ngẫu nhiên** (nếu vẽ cả 1460 hàng thì heatmap quá dày, khó nhìn). `random_state=42` = cố định kết quả ngẫu nhiên để mỗi lần chạy ra giống nhau.

```python
sns.heatmap(sample_matrix.T, cmap='YlOrRd', ...)
```

| Tham số | Ý nghĩa |
|---|---|
| `.T` | **Transpose** — xoay ma trận (hàng↔cột) để cột dữ liệu = hàng heatmap |
| `cmap='YlOrRd'` | Colormap: vàng → cam → đỏ (trắng=0=có data, đỏ=1=missing) |
| `linewidths=0` | Không vẽ đường kẻ ô (vì 200 cột sẽ quá chật) |

---

### 📋 Chiến Lược Xử Lý Missing (Sẽ Thực Hiện ở Buổi 4)

| 🏷️ Cột | 📊 % Missing | 💡 Ý nghĩa thực tế | 🔧 Xử lý |
|---|---|---|---|
| `PoolQC` | 99.5% | Nhà không có bể bơi | Điền `"None"` |
| `MiscFeature` | 96.3% | Không có tiện ích đặc biệt | Điền `"None"` |
| `Alley` | 93.8% | Không có hẻm tiếp cận | Điền `"None"` |
| `Fence` | 80.8% | Không có hàng rào | Điền `"None"` |
| `FireplaceQu` | 47.3% | Không có lò sưởi | Điền `"None"` |
| `GarageType/YrBlt/...` | ~5% | Không có garage | Điền `"None"` / `0` |
| `BsmtQual/Cond/...` | ~2-3% | Không có tầng hầm | Điền `"None"` / `0` |
| `LotFrontage` | 17.7% | Dữ liệu thực sự thiếu | Điền **median theo Neighborhood** |
| `MasVnrType/Area` | ~0.5% | Thiếu ít | Điền mode / `0` |
| `Electrical` | 0.07% | Thiếu 1 hàng | Điền mode |
###
> ⚠️ **Lưu ý quan trọng:** Chúng ta chỉ **phân tích** ở đây, chưa điền! Việc xử lý thực tế sẽ làm ở Buổi 4 để giữ dữ liệu gốc (`train_df`) sạch cho EDA.

**🎯 Mục đích sau cùng của Task 1:** Biết chính xác **19 cột** bị missing, hiểu lý do từng cột (có ý nghĩa vs thực sự thiếu), và có sẵn kế hoạch xử lý → không bị bất ngờ khi chạy mô hình ở Buổi 5.

---

## 📈 Task 2 — Phân Tích Đặc Trưng Số (Numerical Features)

---

### 🤔 Numerical Features Là Gì?

**Numerical features** (đặc trưng số) là các cột chứa con số có thể dùng trực tiếp trong tính toán:

```
Ví dụ trong dataset nhà:
  GrLivArea   = 1710  → Diện tích sống (sqft)
  OverallQual =    8  → Chất lượng tổng thể (1-10)
  YearBuilt   = 2003  → Năm xây dựng
  SalePrice   = 208500 → Giá bán ($)
```

Dataset có **36 numerical features** — nhiều hơn categorical (43 cột). Chúng ta cần biết:
1. Cột nào **ảnh hưởng mạnh nhất** đến giá nhà?
2. Các cột số có **phân phối bất thường** không?
3. Có **outliers** (điểm ngoại lệ) gây nhiễu không?

---

### 📐 Hệ Số Tương Quan Pearson — Công Thức & Ý Nghĩa

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

Trong đó:
- $x_i$ = giá trị của feature (VD: GrLivArea)
- $y_i$ = SalePrice của ngôi nhà đó
- $\bar{x}, \bar{y}$ = giá trị trung bình

| 📊 Giá trị r | 🏷️ Mức độ | 🔍 Ví dụ thực tế |
|---|---|---|
| 0.7 → 1.0 | 🟢 **Tương quan mạnh dương** | Diện tích lớn → Giá cao |
| 0.4 → 0.7 | 🟡 Tương quan trung bình dương | Số phòng ngủ → Giá cao hơn |
| 0.0 → 0.4 | ⚪ Tương quan yếu | Năm sửa chữa → ít ảnh hưởng |
| -0.4 → 0.0 | 🔵 Tương quan yếu âm | — |
| -1.0 → -0.4 | 🔴 Tương quan mạnh âm | Nhà càng cũ → Giá càng thấp |
###
> 💡 **Lưu ý:** r chỉ đo quan hệ **tuyến tính**. Nếu quan hệ cong (phi tuyến), r có thể thấp dù thực ra có liên quan mạnh. Đó là lý do chúng ta cần scatter plots để kiểm tra trực quan!

In [ ]:
# ============================================================
# 📈 TASK 2.1 — TƯƠNG QUAN (CORRELATION) VỚI SALEPRICE
# ============================================================

# --- Bước 1: Tính tương quan với SalePrice ---
num_df    = train_df[numerical_cols + ['SalePrice']].copy()
corr_all  = num_df.corr()['SalePrice'].drop('SalePrice')   # Bỏ hàng SalePrice↔SalePrice=1.0
corr_abs  = corr_all.abs().sort_values(ascending=False)
corr_top  = corr_all[corr_abs.index]                       # Sắp xếp theo |r| giảm dần

print("🏆 Top 15 Features tương quan mạnh nhất với SalePrice:")
print("─" * 50)
for i, (col, r) in enumerate(corr_top.head(15).items(), 1):
    bar   = '█' * int(abs(r) * 20)
    sign  = '+' if r > 0 else '-'
    color = '🟢' if r > 0.5 else '🟡' if r > 0.3 else '🔵'
    print(f"  {i:>2}. {col:<20} {color} r={sign}{abs(r):.3f}  {bar}")

print("\n⚠️ Features tương quan ÂM (có thể gây nhầm lẫn):")
neg_corr = corr_top[corr_top < 0].head(5)
for col, r in neg_corr.items():
    print(f"   {col:<20} r={r:.3f}")

# --- Bước 2: Bar chart 20 features tương quan cao nhất ---
top20 = corr_top.head(20)
colors_bar = ['#2ecc71' if v > 0 else '#e74c3c' for v in top20.values]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(range(len(top20)), top20.values, color=colors_bar, edgecolor='white', alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20.index, fontsize=10)
ax.set_xlabel('Hệ số tương quan Pearson (r)', fontsize=11)
ax.set_title('📊 Top 20 Numerical Features — Tương Quan Với SalePrice', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.axvline(x=0.5, color='green', linestyle='--', alpha=0.5, label='r=0.5 (tương quan mạnh)')
ax.axvline(x=-0.5, color='red', linestyle='--', alpha=0.5)

# Thêm nhãn r= lên thanh
for i, (v, bar) in enumerate(zip(top20.values, bars)):
    ax.text(v + 0.01 if v >= 0 else v - 0.01, i,
            f'{v:.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)

ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**Expected Output:**

![img3-5](images/img_buoi3/img3-5.png)
![img3-6](images/img_buoi3/img3-6.png)

---

### 🔬 Giải Thích Code Tương Quan

---

```python
corr_all = num_df.corr()['SalePrice'].drop('SalePrice')
```

| Bước | Lệnh | Kết quả |
|---|---|---|
| 1 | `num_df.corr()` | Ma trận tương quan 36×36 (tất cả với tất cả) |
| 2 | `['SalePrice']` | Chỉ lấy **cột SalePrice** — tức là r của mỗi feature với SalePrice |
| 3 | `.drop('SalePrice')` | Bỏ hàng `SalePrice↔SalePrice = 1.0` (luôn = 1, không có ý nghĩa) |

```python
corr_abs = corr_all.abs().sort_values(ascending=False)
```

| Lệnh | Ý nghĩa |
|---|---|
| `.abs()` | Lấy giá trị tuyệt đối (để -0.6 xếp ngang 0.6 về mức độ ảnh hưởng) |
| `.sort_values(ascending=False)` | Sắp xếp giảm dần — feature mạnh nhất lên đầu |

---

### 📊 Kết Quả Kỳ Vọng (Giá Trị Thực Từ Dataset)

| # | Feature | r | Ý nghĩa thực tế |
|---|---|---|---|
| 1 | `OverallQual` | **+0.791** | Chất lượng tổng thể (1–10) — quan trọng nhất |
| 2 | `GrLivArea` | **+0.709** | Diện tích sống (sqft) |
| 3 | `GarageCars` | **+0.640** | Số xe garage chứa được |
| 4 | `GarageArea` | **+0.623** | Diện tích garage |
| 5 | `TotalBsmtSF` | **+0.614** | Diện tích tầng hầm |
| 6 | `1stFlrSF` | **+0.606** | Diện tích tầng 1 |
| 7 | `FullBath` | **+0.561** | Số phòng tắm đầy đủ |
| 8 | `TotRmsAbvGrd` | **+0.534** | Tổng số phòng (trừ tắm) |
| 9 | `YearBuilt` | **+0.523** | Năm xây dựng |
| 10 | `YearRemodAdd` | **+0.507** | Năm sửa chữa |

> 💡 **Insight quan trọng:** `OverallQual` (đánh giá chất lượng) quan trọng hơn `GrLivArea` (diện tích) — điều này cho thấy **chất lượng xây dựng > diện tích** trong thị trường BĐS Mỹ. Ở Việt Nam, ngược lại, diện tích thường là yếu tố số 1.

**🎯 Mục đích sau cùng:** Xác định **top 10 features số** quan trọng nhất để tập trung phân tích sâu hơn và ưu tiên khi xây dựng mô hình.

In [ ]:
# ============================================================
# 🔍 TASK 2.2 — SCATTER PLOTS + PHÁT HIỆN OUTLIERS
# ============================================================

# Top 4 features số quan trọng nhất
top4_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF']
top4_corr     = corr_all[top4_features]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('🔍 Scatter Plot: Top 4 Numerical Features vs SalePrice', fontsize=14, fontweight='bold')

for ax, feat in zip(axes.flatten(), top4_features):
    r_val   = corr_all[feat]
    x_data  = train_df[feat]
    y_data  = train_df['SalePrice']

    # Phát hiện outliers (nếu có)
    outlier_mask = (x_data > x_data.quantile(0.995)) | (y_data > y_data.quantile(0.995))

    # Vẽ điểm bình thường
    ax.scatter(x_data[~outlier_mask], y_data[~outlier_mask],
               alpha=0.4, s=15, color='steelblue', label='Normal')
    # Vẽ outliers đỏ
    if outlier_mask.any():
        ax.scatter(x_data[outlier_mask], y_data[outlier_mask],
                   alpha=0.9, s=40, color='red', marker='x', label=f'Outlier ({outlier_mask.sum()})')

    # Thêm text hệ số tương quan
    ax.text(0.05, 0.93, f'r = {r_val:.3f}', transform=ax.transAxes,
            fontsize=11, fontweight='bold',
            color='darkgreen' if abs(r_val) > 0.5 else 'darkorange',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

    ax.set_xlabel(feat, fontsize=10)
    ax.set_ylabel('SalePrice ($)', fontsize=10)
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    if outlier_mask.any():
        ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# --- Phân tích outliers GrLivArea ---
print("\n⚠️ OUTLIERS ĐÁng CHÚ Ý — GrLivArea (Diện tích sống):")
print("─" * 60)
outlier_grliv = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)]
print(f"Số nhà diện tích >4000 sqft nhưng giá <$300,000: {len(outlier_grliv)}")
if len(outlier_grliv) > 0:
    print(outlier_grliv[['GrLivArea', 'SalePrice', 'OverallQual', 'Neighborhood']].to_string())
    print()
    print("💡 Phân tích: Những ngôi nhà này có diện tích rất lớn nhưng giá rất thấp")
    print("   → Có thể là giao dịch đặc biệt (bán lại nội bộ, tài sản thừa kế, v.v.)")
    print("   → Sẽ XEM XÉT loại bỏ ở Buổi 4 để tránh làm nhiễu mô hình")

**Expected Output:**

![img3-7](images/img_buoi3/img3-7.png)
![img3-8](images/img_buoi3/img3-8.png)

---

### 🔬 Giải Thích Code Scatter Plot + Outlier Detection

---

#### 📌 Vòng Lặp Vẽ 4 Subplots

```python
for ax, feat in zip(axes.flatten(), top4_features):
```

| Lệnh | Ý nghĩa |
|---|---|
| `axes.flatten()` | Chuyển mảng 2D `[[ax1,ax2],[ax3,ax4]]` thành list 1D `[ax1,ax2,ax3,ax4]` |
| `zip(axes.flatten(), top4_features)` | Ghép từng ax với từng feature: `(ax1, 'OverallQual')`, `(ax2, 'GrLivArea')`, ... |
| `for ax, feat in zip(...)` | Mỗi lần lặp nhận 1 subplot + 1 tên feature |

---

#### 📌 Phát Hiện Outliers

```python
outlier_mask = (x_data > x_data.quantile(0.995)) | (y_data > y_data.quantile(0.995))
```

| Thành phần | Ý nghĩa |
|---|---|
| `x_data.quantile(0.995)` | Ngưỡng 99.5th percentile của X — chỉ 0.5% dữ liệu vượt qua |
| `\|` (OR) | Bất kỳ điều kiện nào đúng → đánh dấu outlier |
| `outlier_mask` | Series boolean: `True` = outlier, `False` = bình thường |
| `x_data[~outlier_mask]` | `~` = NOT — lấy những điểm **không phải** outlier |

---

#### 📌 Text Hệ Số Tương Quan

```python
ax.text(0.05, 0.93, f'r = {r_val:.3f}', transform=ax.transAxes, ...)
```

| Tham số | Ý nghĩa |
|---|---|
| `0.05, 0.93` | Vị trí X=5% từ trái, Y=93% từ dưới **của subplot** |
| `transform=ax.transAxes` | Tọa độ theo tỷ lệ subplot (0→1), không phải đơn vị dữ liệu |
| `f'r = {r_val:.3f}'` | Format string: hiển thị r với 3 chữ số thập phân |
| `bbox=dict(boxstyle='round', ...)` | Vẽ hộp bo tròn xung quanh text |

```python
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
```

Format trục Y thành tiền: `208500` → `$208,500` (dấu phẩy ngăn nhóm 3 chữ số)

---

#### 📌 Tại Sao Phải Xử Lý Outliers?

```
Ví dụ GrLivArea: 2 ngôi nhà có diện tích >4,000 sqft nhưng giá chỉ ~$130,000
  → Mô hình học: "nhà lớn = giá rẻ" ← SAI!
  → Thực tế: đây là giao dịch đặc biệt, không đại diện thị trường

Nếu giữ outliers:
  RMSE(log) có thể tăng từ 0.12 → 0.15+ chỉ vì 2 điểm dữ liệu này
```

> ⚠️ **Nguyên tắc an toàn:** Chỉ xóa outlier nếu có **lý do rõ ràng** (giao dịch bất thường, lỗi nhập liệu). Không xóa chỉ vì "nó trông lạ"!

**🎯 Mục đích sau cùng:** Hiểu trực quan mối quan hệ giữa từng feature và SalePrice, phát hiện 2 outliers nguy hiểm trong GrLivArea → đưa ra quyết định xóa có căn cứ ở Buổi 4.

---

## 🏷️ Task 3 — Phân Tích Đặc Trưng Phân Loại (Categorical Features)

---

### 🤔 Categorical Features Là Gì?

**Categorical features** (đặc trưng phân loại) là các cột chứa **nhãn/chữ** thay vì con số:

```
Ví dụ trong dataset nhà:
  Neighborhood = "CollgCr"   → Khu vực địa lý
  HouseStyle   = "2Story"    → Kiểu nhà (1 tầng, 2 tầng, ...)
  KitchenQual  = "Ex"        → Chất lượng bếp (Ex=Excellent, Gd=Good, ...)
  SaleType     = "WD"        → Loại giao dịch (WD=Warranty Deed thông thường)
```

Dataset có **43 categorical features**. Vấn đề: **mô hình ML không thể tính toán với chữ!**

---

### 🔄 Tại Sao Categorical Features Cần Được Mã Hóa (Encoding)?

```
Train data hiện tại:
  Neighborhood = ["CollgCr", "Veenker", "Crawfor", ...]
  KitchenQual  = ["Gd", "TA", "Ex", "Gd", ...]

Mô hình Linear Regression nhìn vào và hỏi: "CollgCr + Veenker = ?"  ← Không tính được!

Sau encoding (sẽ làm ở Buổi 4):
  Neighborhood_CollgCr = [1, 0, 0, ...]
  Neighborhood_Veenker = [0, 1, 0, ...]
  KitchenQual_ordinal  = [3, 2, 4, 3, ...]  ← Tốt=3, TB=2, Xuất sắc=4
```

---

### 📊 Hai Loại Categorical trong Dataset Này

| Loại | Ví dụ cột | Đặc điểm | Encoding tốt nhất |
|---|---|---|---|
| **Nominal** (không có thứ tự) | Neighborhood, HouseStyle, SaleType | "CollgCr" không hơn "Veenker" | One-Hot Encoding |
| **Ordinal** (có thứ tự rõ ràng) | KitchenQual (Po<Fa<TA<Gd<Ex), OverallQual | Excellent > Good > Fair | Ordinal Encoding |

> 💡 **Cardinality** = số giá trị duy nhất trong một cột categorical:
> - `Neighborhood`: 25 giá trị duy nhất → **High cardinality** (nếu One-Hot: thêm 25 cột!)
> - `CentralAir`: 2 giá trị (Y/N) → **Low cardinality** (One-Hot chỉ thêm 1 cột)

In [ ]:
# ============================================================
# 🏷️ TASK 3.1 — PHÂN TÍCH CATEGORICAL: OVERVIEW + NEIGHBORHOOD
# ============================================================

cat_df = train_df[categorical_cols + ['SalePrice']].copy()

# --- Bước 1: Overview — Cardinality từng cột ---
cardinality = cat_df[categorical_cols].nunique().sort_values(ascending=False)
print("📊 Cardinality (số giá trị duy nhất) của 43 Categorical Features:")
print("─" * 60)
high_card  = cardinality[cardinality > 10]
mid_card   = cardinality[(cardinality > 3) & (cardinality <= 10)]
low_card   = cardinality[cardinality <= 3]
print(f"  🔴 High cardinality (>10 giá trị) : {len(high_card)} cột → {list(high_card.index)}")
print(f"  🟡 Mid cardinality (4-10 giá trị) : {len(mid_card)} cột")
print(f"  🟢 Low cardinality (≤3 giá trị)   : {len(low_card)} cột")

# --- Bước 2: Biểu đồ Neighborhood vs SalePrice ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🏙️ Phân Tích Categorical Features', fontsize=14, fontweight='bold')

# Tính median SalePrice theo Neighborhood và sắp xếp
neighborhood_price = (train_df.groupby('Neighborhood')['SalePrice']
                      .median()
                      .sort_values(ascending=True))

# Màu gradient: xanh lá (rẻ) → đỏ (đắt)
n_colors   = len(neighborhood_price)
color_vals = plt.cm.RdYlGn(np.linspace(0, 1, n_colors))

axes[0].barh(neighborhood_price.index, neighborhood_price.values,
             color=color_vals, edgecolor='white')
axes[0].set_xlabel('Median SalePrice ($)', fontsize=10)
axes[0].set_title('📍 Median Giá Nhà theo Neighborhood', fontsize=11, fontweight='bold')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Thêm nhãn giá
for i, (hood, price) in enumerate(neighborhood_price.items()):
    axes[0].text(price + 500, i, f'${price/1000:.0f}K', va='center', fontsize=7)

# --- Bước 3: OverallQual Box Plot ---
qual_groups = [train_df[train_df['OverallQual'] == q]['SalePrice'].values
               for q in sorted(train_df['OverallQual'].unique())]
qual_labels = sorted(train_df['OverallQual'].unique())

bp = axes[1].boxplot(qual_groups, labels=qual_labels, patch_artist=True, notch=False)

# Màu hộp: xanh lam → xanh lá
box_colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(qual_groups)))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)

axes[1].set_xlabel('OverallQual (1=Tệ → 10=Xuất sắc)', fontsize=10)
axes[1].set_ylabel('SalePrice ($)', fontsize=10)
axes[1].set_title('📦 SalePrice theo OverallQual', fontsize=11, fontweight='bold')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

# --- Summary Neighborhood extremes ---
print("\n🏙️ PHÂN TÍCH NEIGHBORHOOD:")
print(f"  💰 Top 5 khu vực đắt nhất  : {list(neighborhood_price.tail(5).index)}")
print(f"  🏚️  Top 5 khu vực rẻ nhất   : {list(neighborhood_price.head(5).index)}")
print(f"\n  Chênh lệch: ${neighborhood_price.min():,.0f} → ${neighborhood_price.max():,.0f}")
print(f"  Tỷ lệ: {neighborhood_price.max()/neighborhood_price.min():.1f}x  ← Khu vực đắt gấp ~{neighborhood_price.max()/neighborhood_price.min():.0f}x khu rẻ!")

**Expected Output:**

![img3-9](images/img_buoi3/img3-9.png)
![img3-10](images/img_buoi3/img3-10.png)

---

### 🔬 Giải Thích Code Categorical Analysis

---

#### 📌 Cardinality — `nunique()`

```python
cardinality = cat_df[categorical_cols].nunique().sort_values(ascending=False)
```

| Lệnh | Ý nghĩa |
|---|---|
| `.nunique()` | **Number of Unique** — đếm bao nhiêu giá trị khác nhau trong mỗi cột |
| Ví dụ | `Neighborhood.nunique()` = 25 (có 25 khu vực khác nhau) |

---

#### 📌 GroupBy + Median

```python
neighborhood_price = (train_df.groupby('Neighborhood')['SalePrice']
                      .median()
                      .sort_values(ascending=True))
```

| Bước | Lệnh | Kết quả |
|---|---|---|
| 1 | `.groupby('Neighborhood')` | Nhóm 1460 nhà thành 25 nhóm theo khu vực |
| 2 | `['SalePrice']` | Lấy cột SalePrice của mỗi nhóm |
| 3 | `.median()` | Tính giá trung vị của mỗi khu vực |
| 4 | `.sort_values(ascending=True)` | Sắp xếp từ rẻ → đắt (để barh đọc dễ) |

**Tại sao dùng `median` thay vì `mean`?** Vì median ít bị ảnh hưởng bởi outliers — 1 ngôi nhà cực kỳ đắt trong khu vực rẻ sẽ không kéo lệch kết quả.

---

#### 📌 Gradient Color — `plt.cm.RdYlGn(np.linspace(0, 1, n))`

```python
color_vals = plt.cm.RdYlGn(np.linspace(0, 1, n_colors))
```

| Thành phần | Ý nghĩa |
|---|---|
| `plt.cm.RdYlGn` | Colormap: **R**ed-**Y**e**l**low-**G**reen (đỏ→vàng→xanh) |
| `np.linspace(0, 1, n)` | Tạo n số từ 0.0 → 1.0 đều nhau |
| Kết quả | 25 màu: đỏ nhạt (khu rẻ) → xanh lá (khu đắt) |

---

#### 📌 Box Plot — `boxplot()`

```python
bp = axes[1].boxplot(qual_groups, labels=qual_labels, patch_artist=True)
```

| Tham số | Ý nghĩa |
|---|---|
| `qual_groups` | List các mảng giá trị SalePrice, nhóm theo OverallQual |
| `patch_artist=True` | Cho phép tô màu hộp (mặc định không tô) |
| Hộp (box) | 25th – 75th percentile (50% dữ liệu giữa) |
| Đường giữa | Median (50th percentile) |
| Râu (whisker) | ±1.5×IQR |
| Chấm đỏ | Outliers ngoài whisker |

---

### 📊 Kết Quả Kỳ Vọng

**Neighborhood (giá median):**
```
Rẻ nhất:  MeadowV ~$98K, BrDale ~$106K, IDOTRR ~$100K
Đắt nhất: NridgHt ~$315K, NoRidge ~$301K, StoneBr ~$278K
Chênh lệch: ~3x giữa khu đắt và rẻ nhất
```

**OverallQual vs SalePrice:**
```
OverallQual 1–3: $40K–$100K  (nhà chất lượng kém)
OverallQual 5–6: $140K–$190K (nhà trung bình)
OverallQual 8–10: $270K–$500K+ (nhà cao cấp)
→ Xu hướng tăng đơn điệu và rõ ràng → Feature tốt nhất!
```

**🎯 Mục đích sau cùng:** Chứng minh Neighborhood và OverallQual là 2 categorical features cực kỳ quan trọng → sẽ cần xử lý encoding cẩn thận ở Buổi 4.

---

## 🌡️ Task 4 — Ma Trận Tương Quan & Đa Cộng Tuyến (Multicollinearity)

---

### 🤔 Ma Trận Tương Quan Là Gì?

**Correlation matrix** = Bảng hiển thị hệ số tương quan Pearson **giữa TẤT CẢ các cặp features** cùng lúc:

```
            GrLivArea  GarageCars  TotalBsmtSF  SalePrice
GrLivArea      1.000       0.468        0.454      0.709
GarageCars     0.468       1.000        0.435      0.640
TotalBsmtSF    0.454       0.435        1.000      0.614
SalePrice      0.709       0.640        0.614      1.000
```

Đường chéo = 1.0 (mỗi biến tương quan 100% với chính nó)
Bảng đối xứng qua đường chéo (A↔B = B↔A)

---

### ⚠️ Đa Cộng Tuyến (Multicollinearity) Là Gì? Tại Sao Nguy Hiểm?

**Multicollinearity** = khi hai hay nhiều features **tương quan cao với nhau** (|r| > 0.8):

```
Ví dụ nguy hiểm trong dataset nhà:
  GarageCars ↔ GarageArea  :  r = 0.882  ← RẤT CAO!
  (Số xe chứa được ↔ Diện tích garage — đương nhiên liên quan!)

  GrLivArea ↔ TotRmsAbvGrd :  r = 0.825
  (Nhà rộng thì nhiều phòng — hiển nhiên!)
```

**Tại sao nguy hiểm với Linear Regression?**

```
Mô hình muốn học:
  SalePrice = a×GarageCars + b×GarageArea + ...

Nhưng GarageCars ≈ GarageArea × (hằng số)
→ Mô hình KHÔNG BIẾT phân bổ trọng số a và b như thế nào
→ Hệ số a, b dao động lớn, không ổn định
→ Dự đoán trên dữ liệu mới bị sai lệch (overfitting)
```

> ✅ **Giải pháp:** Loại bỏ một trong hai features thừa, hoặc dùng **Regularization** (Ridge/Lasso — sẽ học ở Buổi 6)

In [ ]:
# ============================================================
# 🌡️ TASK 4.1 — CORRELATION HEATMAP & MULTICOLLINEARITY
# ============================================================

# --- Bước 1: Chọn top 15 features + SalePrice ---
top15_cols = corr_abs.head(15).index.tolist()
heatmap_df = train_df[top15_cols + ['SalePrice']].copy()
corr_matrix = heatmap_df.corr()

# --- Bước 2: Mask tam giác trên (chỉ hiện nửa dưới) ---
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# --- Bước 3: Vẽ heatmap ---
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    corr_matrix,
    mask     = mask,
    annot    = True,
    fmt      = '.2f',
    cmap     = 'RdYlGn',
    center   = 0,
    vmin     = -1,
    vmax     = 1,
    linewidths = 0.5,
    linecolor  = 'white',
    cbar_kws   = {'shrink': 0.8, 'label': 'Pearson r'},
    ax       = ax
)
ax.set_title('🌡️ Ma Trận Tương Quan — Top 15 Numerical Features + SalePrice',
             fontsize=13, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
plt.tight_layout()
plt.show()

# --- Bước 4: Phát hiện cặp đa cộng tuyến (|r| > 0.75, không tính SalePrice) ---
print("\n⚠️ PHÁT HIỆN ĐA CỘNG TUYẾN (|r| > 0.75 giữa các FEATURES):")
print("─" * 65)

multicollinear_pairs = []
cols = [c for c in corr_matrix.columns if c != 'SalePrice']

for i, col_a in enumerate(cols):
    for col_b in cols[i+1:]:
        r_val = corr_matrix.loc[col_a, col_b]
        if abs(r_val) > 0.75:
            multicollinear_pairs.append((col_a, col_b, r_val))

multicollinear_pairs.sort(key=lambda x: abs(x[2]), reverse=True)

for col_a, col_b, r_val in multicollinear_pairs:
    severity = "🔴 Nghiêm trọng" if abs(r_val) > 0.85 else "🟠 Đáng chú ý"
    print(f"  {severity}  {col_a:<20} ↔ {col_b:<20}  r = {r_val:.3f}")

print()
print("💡 QUYẾT ĐỊNH XỬ LÝ (sẽ thực hiện ở Buổi 4):")
decisions = [
    ("GarageCars", "GarageArea",   "GarageCars", "Dùng số xe thay diện tích (trực quan hơn)"),
    ("GrLivArea",  "TotRmsAbvGrd", "GrLivArea",  "Diện tích chứa nhiều thông tin hơn số phòng"),
    ("TotalBsmtSF","1stFlrSF",     "TotalBsmtSF","Tổng hầm bao gồm 1stFlrSF trong nhiều trường hợp"),
]
for col_a, col_b, keep, reason in decisions:
    print(f"  ✅ Giữ [{keep}] | ❌ Xem xét drop [{col_b if keep==col_a else col_a}] → {reason}")

**Expected Output:**

![img3-11](images/img_buoi3/img3-11.png)
![img3-12](images/img_buoi3/img3-12.png)

---

### 🔬 Giải Thích Code Correlation Heatmap

---

#### 📌 Tạo Mask Tam Giác

```python
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
```

| Lệnh | Ý nghĩa |
|---|---|
| `np.ones_like(corr_matrix)` | Tạo ma trận toàn số 1, cùng shape với `corr_matrix` |
| `dtype=bool` | Chuyển 1 → `True` |
| `np.triu(...)` | **Upper Triangle** — giữ `True` ở tam giác trên (bao gồm đường chéo) |
| `mask=mask` | Seaborn sẽ **ẩn** các ô có mask=True → chỉ hiện tam giác dưới |

**Tại sao chỉ hiện nửa dưới?** Vì ma trận tương quan đối xứng — `r(A,B) = r(B,A)`. Hiện cả 2 nửa thừa và khó đọc.

---

#### 📌 Tham Số `sns.heatmap()`

```python
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ...)
```

| Tham số | Ý nghĩa |
|---|---|
| `annot=True` | **Annotate** — hiện số r trong mỗi ô |
| `fmt='.2f'` | Format số: 2 chữ số thập phân (0.71 thay vì 0.7090...) |
| `cmap='RdYlGn'` | Colormap: đỏ=-1 (tương quan âm mạnh), vàng=0, xanh=+1 |
| `center=0` | Căn giữa màu vàng tại r=0 |
| `vmin=-1, vmax=1` | Giới hạn scale màu từ -1 đến +1 |
| `linewidths=0.5` | Đường kẻ mỏng giữa các ô để dễ phân biệt |

---

### 📊 Kết Quả Kỳ Vọng — Cặp Đa Cộng Tuyến

| 🔴/🟠 | Feature A | Feature B | r | Xử lý |
|---|---|---|---|---|
| 🔴 | `GarageCars` | `GarageArea` | **0.882** | Drop `GarageArea` |
| 🔴 | `YearBuilt` | `GarageYrBlt` | **0.826** | Drop `GarageYrBlt` |
| 🔴 | `GrLivArea` | `TotRmsAbvGrd` | **0.825** | Drop `TotRmsAbvGrd` |
| 🟠 | `TotalBsmtSF` | `1stFlrSF` | **0.820** | Xem xét drop `1stFlrSF` |

> 💡 **Lưu ý:** Random Forest và XGBoost ít bị ảnh hưởng bởi multicollinearity hơn Linear Regression, vì chúng sử dụng feature importance chứ không phải hệ số tuyến tính. Nhưng loại bỏ vẫn tốt cho tốc độ training!

**🎯 Mục đích sau cùng:** Xác định chính xác các cặp features thừa → lên kế hoạch loại bỏ ở Buổi 4, giúp mô hình Linear Regression hoạt động ổn định hơn.

---

## 📝 BÀI TẬP BUỔI 3 — Tự Luyện EDA

> 🎯 **Mục tiêu:** Củng cố kỹ năng EDA — phân tích missing values, tương quan, và trực quan hoá dữ liệu.
> Hãy tự làm trước, sau đó xem gợi ý nếu bí!

---

### 🟢 Bài 1 (Dễ) — Phân Tích Missing Values Của `test_df`

**Yêu cầu:** Ở Task 1 chúng ta đã phân tích missing values của `train_df`.  
Hãy làm tương tự cho **`test_df`** và trả lời:

1. `test_df` có bao nhiêu cột bị missing?
2. Cột nào missing nhiều nhất? Bao nhiêu phần trăm?
3. So với `train_df` (19 cột), `test_df` nhiều hay ít cột missing hơn?

> 💡 **Gợi ý:** Dùng lại đúng code từ Task 1, chỉ thay `train_df` → `test_df`

In [ ]:
# ============================================================
# 📝 BÀI TẬP 1 — Phân tích missing values của test_df
# ============================================================
# Hướng dẫn: Điền code vào chỗ có dấu ???
# Chạy cell này để kiểm tra kết quả của bạn
# ============================================================

# Bước 1: Đếm missing trong test_df
missing_test_count = ???.isnull().sum()             # TODO: thay ??? bằng tên DataFrame test

# Bước 2: Tính phần trăm
missing_test_pct = (missing_test_count / len(???)) * 100  # TODO: thay ??? 

# Bước 3: Tạo DataFrame tổng hợp
missing_test_df = pd.DataFrame({
    'missing_count': missing_test_count,
    'missing_pct'  : missing_test_pct
})
missing_test_df = missing_test_df[missing_test_df['missing_count'] > ???]  # TODO: lọc cột có missing
missing_test_df = missing_test_df.sort_values('???', ascending=False)       # TODO: sắp xếp theo %

# Bước 4: In kết quả
print(f"test_df có {???} cột bị missing")             # TODO: điền số lượng
print(f"Cột missing nhiều nhất: {missing_test_df.index[0]}, {missing_test_df['missing_pct'].iloc[0]:.1f}%")
print()
print(missing_test_df.head(10))

**Expected Output:**

![img3-13](images/img_buoi3/img3-13.png)

#### 🔑 Đáp án Bài 1 (mở ra sau khi tự làm!)

<details>
<summary>👆 Click để xem đáp án</summary>

```python
missing_test_count = test_df.isnull().sum()
missing_test_pct   = (missing_test_count / len(test_df)) * 100

missing_test_df = pd.DataFrame({
    'missing_count': missing_test_count,
    'missing_pct'  : missing_test_pct
})
missing_test_df = missing_test_df[missing_test_df['missing_count'] > 0]
missing_test_df = missing_test_df.sort_values('missing_pct', ascending=False)

print(f"test_df có {len(missing_test_df)} cột bị missing")
print(f"Cột missing nhiều nhất: {missing_test_df.index[0]}, {missing_test_df['missing_pct'].iloc[0]:.1f}%")
print()
print(missing_test_df.head(10))
```

**Kết quả kỳ vọng:**
```
test_df có 33 cột bị missing   ← Nhiều hơn train_df (19 cột) vì test set đa dạng hơn!
Cột missing nhiều nhất: PoolQC, 99.79%
```
</details>

---

### 🟡 Bài 2 (Trung bình) — Tương Quan Của Các Feature "Năm"

**Yêu cầu:** Dataset có nhiều cột liên quan đến **năm** (`YearBuilt`, `YearRemodAdd`, `GarageYrBlt`, `MoSold`, `YrSold`).

1. Lọc ra tất cả cột numerical có chữ `"Year"` hoặc `"Yr"` trong tên
2. Tính hệ số tương quan của từng cột đó với `SalePrice`
3. Vẽ bar chart nằm ngang thể hiện kết quả
4. Nhận xét: Cột nào quan trọng nhất? Cột nào ít quan trọng nhất?

> 💡 **Gợi ý:** Dùng list comprehension `[col for col in numerical_cols if 'Year' in col or 'Yr' in col]`

In [ ]:
# ============================================================
# 📝 BÀI TẬP 2 — Tương quan của các feature "Năm"
# ============================================================

# Bước 1: Lọc ra cột có "Year" hoặc "Yr" trong tên
year_cols = [col for col in ??? if '???' in col or '???' in col]  # TODO: điền vào
print("Các cột liên quan đến năm:", year_cols)
print()

# Bước 2: Tính tương quan với SalePrice
year_corr = train_df[year_cols].corrwith(train_df['???'])  # TODO: tên cột target
year_corr = year_corr.sort_values(ascending=???)           # TODO: True hay False để xem từ cao → thấp?

print("Hệ số tương quan với SalePrice:")
print(year_corr.round(3))

# Bước 3: Vẽ bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in year_corr.values]
ax.barh(year_corr.index, year_corr.values, color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Hệ số tương quan Pearson (r)')
ax.set_title('📅 Tương Quan của Các Feature "Năm" với SalePrice')
plt.tight_layout()
plt.show()

# Bước 4: Nhận xét — điền vào đây!
print("\n💬 Nhận xét của bạn:")
print("   Cột quan trọng nhất : ???")
print("   Cột ít quan trọng   : ???")
print("   Tại sao MoSold/YrSold có r thấp? : ???")

**Expected Output:**

![img3-14](images/img_buoi3/img3-14.png)

#### 🔑 Đáp án Bài 2

<details>
<summary>👆 Click để xem đáp án</summary>

```python
year_cols = [col for col in numerical_cols if 'Year' in col or 'Yr' in col]

year_corr = train_df[year_cols].corrwith(train_df['SalePrice'])
year_corr = year_corr.sort_values(ascending=False)
```

**Kết quả kỳ vọng:**
```
YearBuilt      0.523   ← Nhà mới xây → giá cao hơn
YearRemodAdd   0.507   ← Mới sửa chữa → giá cao hơn
GarageYrBlt    0.486
MoSold         0.046   ← Tháng bán gần như không ảnh hưởng giá!
YrSold        -0.029   ← Năm bán cũng không ảnh hưởng (thị trường ổn định 2006-2010)
```

**Nhận xét:**
- `YearBuilt` quan trọng nhất → nhà mới = giá cao
- `MoSold`, `YrSold` gần bằng 0 → thời điểm bán không quyết định giá nhà trong dataset này
- → Có thể **drop** `MoSold` và `YrSold` ở Buổi 4!
</details>

---

### 🔴 Bài 3 (Nâng cao) — Box Plot `KitchenQual` vs `SalePrice`

**Yêu cầu:** `KitchenQual` (Chất lượng bếp) có 4 giá trị: `Po` (Poor), `Fa` (Fair), `TA` (Typical), `Gd` (Good), `Ex` (Excellent).

1. Vẽ **box plot** thể hiện phân phối `SalePrice` theo từng mức `KitchenQual`
2. Sắp xếp trục X theo **thứ tự chất lượng** (Po → Fa → TA → Gd → Ex), **không** theo alphabet
3. Thêm tiêu đề, nhãn trục, và format giá tiền trục Y
4. Nhận xét: Bếp chất lượng `Ex` đắt hơn `TA` bao nhiêu lần (so median)?

> 💡 **Gợi ý:** Dùng `order=['Po','Fa','TA','Gd','Ex']` trong `sns.boxplot()`

In [ ]:
# ============================================================
# 📝 BÀI TẬP 3 — Box Plot KitchenQual vs SalePrice
# ============================================================

# Bước 1: Xem các giá trị unique của KitchenQual
print("Các giá trị KitchenQual:", train_df['KitchenQual'].unique())
print("Số lượng từng loại:\n", train_df['KitchenQual'].value_counts())
print()

# Bước 2: Định nghĩa thứ tự đúng (từ tệ → xuất sắc)
quality_order = [???]   # TODO: điền vào đúng thứ tự: 'Po', 'Fa', 'TA', 'Gd', 'Ex'

# Bước 3: Vẽ box plot
fig, ax = plt.subplots(figsize=(9, 5))

sns.boxplot(
    data  = train_df,
    x     = '???',          # TODO: tên cột categorical
    y     = '???',          # TODO: tên cột giá
    order = ???,             # TODO: dùng biến quality_order ở trên
    palette = 'YlOrRd',
    ax    = ax
)

# Bước 4: Định dạng biểu đồ
ax.set_title('???', fontsize=13, fontweight='bold')   # TODO: đặt tiêu đề
ax.set_xlabel('Chất lượng bếp (Po=Tệ → Ex=Xuất sắc)', fontsize=11)
ax.set_ylabel('SalePrice ($)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

# Bước 5: Tính và so sánh median
for qual in quality_order:
    if qual in train_df['KitchenQual'].values:
        median_price = train_df[train_df['KitchenQual'] == qual]['SalePrice'].median()
        print(f"  {qual}: ${median_price:>10,.0f}")

# TODO: Tính xem Ex đắt hơn TA bao nhiêu lần?
median_ex = train_df[train_df['KitchenQual'] == 'Ex']['SalePrice'].median()
median_ta = train_df[train_df['KitchenQual'] == 'TA']['SalePrice'].median()
print(f"\nBếp 'Ex' đắt hơn 'TA': {???:.1f}x")  # TODO: tính tỷ lệ

**Expected Output:**

![img3-15](images/img_buoi3/img3-15.png)

#### 🔑 Đáp án Bài 3

<details>
<summary>👆 Click để xem đáp án</summary>

```python
quality_order = ['Po', 'Fa', 'TA', 'Gd', 'Ex']

sns.boxplot(
    data    = train_df,
    x       = 'KitchenQual',
    y       = 'SalePrice',
    order   = quality_order,
    palette = 'YlOrRd',
    ax      = ax
)
ax.set_title('🍳 Chất Lượng Bếp vs Giá Nhà', fontsize=13, fontweight='bold')

# Tỷ lệ Ex/TA
print(f"Bếp 'Ex' đắt hơn 'TA': {median_ex/median_ta:.1f}x")
```

**Kết quả kỳ vọng:**
```
Po :  $   84,900   (rất ít nhà)
Fa :  $  106,475
TA :  $  153,000   ← Phổ biến nhất (735 nhà)
Gd :  $  202,000
Ex :  $  285,000

Bếp 'Ex' đắt hơn 'TA': 1.9x  ← Gần gấp đôi chỉ vì chất lượng bếp!
```

**Nhận xét quan trọng:** `KitchenQual` là categorical ordinal — có thứ tự rõ ràng (Po<Fa<TA<Gd<Ex). Ở Buổi 4 sẽ encode thành số: Po=1, Fa=2, TA=3, Gd=4, Ex=5.
</details>

---

> ✅ **Hoàn thành 3 bài tập Buổi 3!** Nếu làm đúng cả 3, bạn đã thực sự hiểu EDA.  
> 🚀 **Tiếp theo:** Buổi 4 — Data Cleaning & Preprocessing (dọn dẹp dựa trên những gì EDA đã tìm ra)

---

## ✅ Tổng Kết Buổi 3 — Exploratory Data Analysis

---

### 🎯 Checklist Những Gì Đã Làm

| ✅ | Nhiệm vụ | Kết quả |
|---|---|---|
| ✅ | Setup lại môi trường | Import đủ thư viện, load train/test data |
| ✅ | Phân tích Missing Values | 19 cột missing, PoolQC 99.5%, kế hoạch xử lý |
| ✅ | Visualize Missing Pattern | Bar chart + Heatmap |
| ✅ | Numerical Correlation | OverallQual (0.791) là feature số 1 |
| ✅ | Scatter Plots + Outliers | 2 outliers nguy hiểm trong GrLivArea |
| ✅ | Categorical Analysis | Neighborhood: chênh lệch 3x giữa khu đắt/rẻ |
| ✅ | Correlation Heatmap | 4 cặp đa cộng tuyến nguy hiểm |

---

### 📊 EDA Findings — Tóm Tắt Phát Hiện Chính

```
🏆 TOP 5 FEATURES QUAN TRỌNG NHẤT:
   1. OverallQual   → r = 0.791  (chất lượng tổng thể)
   2. GrLivArea     → r = 0.709  (diện tích sống)
   3. GarageCars    → r = 0.640  (số xe garage)
   4. TotalBsmtSF   → r = 0.614  (diện tích tầng hầm)
   5. Neighborhood  → ~0.5+ ước tính (khu vực)

⚠️ VẤN ĐỀ CẦN XỬ LÝ Ở BUỔI 4:
   Missing: 19 cột, PoolQC 99.5% → điền "None"
   Outliers: 2 nhà GrLivArea >4000 sqft, giá <$300K
   Multicollinearity: GarageCars↔GarageArea (r=0.882)
```

---

### 🔮 Preview Buổi 4 — Data Cleaning & Preprocessing

Buổi 4 sẽ biến những phát hiện EDA thành hành động thực tế:

| 🔧 Nhiệm vụ | 📝 Chi tiết |
|---|---|
| Xử lý Missing Values | Điền "None"/0 cho cột có nghĩa, median theo nhóm cho LotFrontage |
| Loại bỏ Outliers | Xóa 2 nhà GrLivArea >4000 & SalePrice <$300K |
| Feature Engineering | Tạo TotalSF = GrLivArea + TotalBsmtSF, HouseAge = 2010 - YearBuilt |
| Encoding Categorical | Ordinal cho KitchenQual/ExterQual, One-Hot cho Neighborhood/... |
| Xử lý Multicollinearity | Drop GarageArea, TotRmsAbvGrd, GarageYrBlt |

---

### 📋 Checklist Trước Buổi 4

- [ ] Đã chạy tất cả cells trong Buổi 3 thành công (không có lỗi)
- [ ] Đã xem biểu đồ missing values và hiểu 19 cột bị thiếu
- [ ] Đã nhớ: OverallQual là feature quan trọng nhất
- [ ] Đã thấy 2 outliers đỏ trong scatter plot GrLivArea
- [ ] Đã hiểu tại sao GarageCars và GarageArea không nên dùng cùng lúc

> 🎓 **Bạn đã hoàn thành Buổi 3!** EDA là bước khó nhất vì không có "đáp án đúng" — phải dùng kiến thức domain (bất động sản) + thống kê kết hợp. Buổi 4 sẽ "dọn dẹp" dữ liệu dựa trên những gì đã khám phá hôm nay.

---
*📌 Buổi 3/9 — Khóa học Dự Đoán Giá Nhà bằng Machine Learning (Tiếng Việt)*